In [1]:
import pandas as pd
from pathlib import Path

import json
index_path = Path(r"C:\Users\craig\Documents\CS 125\Juke_Jam\indexes\indexes.json")
profiles_path = Path(r"C:\Users\craig\Documents\CS 125\Juke_Jam\indexes\time_context_profiles.json")


with open(index_path) as f:
    index_data = json.load(f)

genre_index = index_data["genre"]
mood_index = index_data["mood"]
energy_index = index_data["energy"]
artist_index = index_data["artist"]
title_index = index_data["title"]

with open(profiles_path) as f:
    time_profiles = json.load(f)

In [2]:
# Feature Vectors

# Feature set
features = set()

# genre features
for g in genre_index:
    features.add(f"genre:{g}")

# mood features
for m in mood_index:
    features.add(f"mood:{m}")

# energy features
for e in energy_index:
    features.add(f"energy:{e}")

# artist features
for a in artist_index:
    features.add(f"artist:{a}")

# title features
for t in title_index:
    features.add(f"title:{t}")

# for ind in [genre_index, mood_index, energy_index, artist_index, title_index]:
#     features |= set(ind.keys())

# for time_period, profile in time_profiles.items():
#     for j, values in profile.items():
#         for v in values:
#             features.add(f"{j}:{v}")

features = sorted(features)

feature_inds = {f: i for i, f in enumerate(features)}


In [3]:
# Song feature vectors
import numpy as np
from collections import defaultdict

num_features = len(features)
song_vectors = defaultdict(dict)

def update_song_features(song_id, feature):
    song_id = str(song_id)

    if feature not in feature_inds:
        return
        
    song_vectors[song_id][feature_inds[feature]] = 1

In [4]:
# Create song feature vectors
def populate_index(index, prefix=None):
    for feature, songs in index.items():
        f = f"{prefix}:{feature}" if prefix else feature

        if f not in feature_inds:
            print("Missing feature in vocab:", f)
            raise ValueError("Feature mismatch")

        for s in songs:
            update_song_features(s, f)


populate_index(genre_index, "genre")
populate_index(mood_index, "mood")
populate_index(energy_index, "energy")
populate_index(artist_index, "artist")
populate_index(title_index, "title")

In [9]:
# Compute IDF Weights
song_vector_num = len(song_vectors)
idf = {}

df_counts = defaultdict(int)

for vec in song_vectors.values():
    for key in vec.keys():
        df_counts[key] += 1

for f, i in feature_inds.items():
    df = df_counts.get(i, 0)
    idf[f] = np.log(song_vector_num / (1 + df))

# for f in features:
#     df = 0
#     f_index = feature_inds[f]
#     for vec in song_vectors.values():
#         if vec.get(f_index, 0) > 0:
#         # if vec[feature_inds[f]] > 0:
#             df += 1
#     idf[f] = np.log(song_vector_num / (1 + df))


In [10]:
# for song, vec in song_vectors.items():
#     for feature, ind in feature_inds.items():
#         if vec[ind] > 0:
#             vec[ind] *= idf

for song, vec in song_vectors.items():
    for i in list(vec.keys()):
        feature = features[i]
        vec[i] *= idf[feature]

In [12]:
def build_query_vector(genres=None, mood=None, energy=None, artist=None,
                       title=None, time_of_day=None):
    q = {}

    def set_feature(f):
        if f in feature_inds:
            q[feature_inds[f]] = 1
    
    if genres:
        for g in genres:
            set_feature(f"genre:{g}")
    
    if mood:
        set_feature(f"mood:{mood}")
    
    if mood:
        set_feature(f"energy:{energy}")

    if mood:
        set_feature(f"artist:{artist}")
    
    if title:
        for token in title.split():
            set_feature(f"title:{token.lower()}")
    
    # time context boost
    if time_of_day and time_of_day in time_profiles:
        profile = time_profiles[time_of_day]
        for category, values in profile.items():
            for v in values:
                set_feature(f"{category}:{v}")
    
    return q

In [ ]:
def retrieve_candidates(genres=None, mood=None, energy=None,
                        artist=None, title=None):

    sets = []

    def add_hits(index, keys):
        if keys:
            hits = set()
            for k in keys:
                hits |= set(index.get(k, []))
            sets.append(hits)

    add_hits(genre_index, genres)
    add_hits(mood_index, [mood] if mood else None)
    add_hits(energy_index, [energy] if energy else None)
    add_hits(artist_index, [artist] if artist else None)

    if title:
        tokens = title.split()
        add_hits(title_index, tokens)

    if not sets:
        return []

    return list(set.union(*sets))


In [15]:
# Cosine Similarity
# cos(x) = AB/(||A|||B||)
# def cosine_similarity(a, b):
#     n_a = np.linalg.norm(a) == 0
#     n_b = np.linalg.norm(b) == 0
#     if n_a == 0 or n_b == 0:
#         return 0
#     return np.dot(a, b) / (n_a * n_b)

# Cosine sim for sparse vector
def cosine_similarity(q_vec, song_vec):
    dot = 0

    for i, val in song_vec.items():
        if i in q_vec:
            dot += val*q_vec[i]
    
    norm_q = np.sqrt(sum(v*v for v in q_vec.values()))
    norm_song = np.sqrt(sum(v*v for v in song_vec.values()))

    if norm_q == 0 or norm_song == 0:
        return 0
    
    return dot / (norm_q * norm_song)

In [16]:
# Ranking
def rank_songs(query, top_k=10):
    candidates = retrieve_candidates(**query)

    if not candidates:
        return []
    
    query_vec = build_query_vector(**query)

    scored = []
    for s in candidates:
        score = cosine_similarity(query_vec, song_vectors[s])
        scored.append((s, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


In [ ]:
# Ranking Test 1
query = {
    "genres": ["rock"],
    "mood": "",
    "energy": ""
}

results = rank_songs(query, top_k=5)

print("Results:", results)
print("Returned:", len(results))

Results: []
Returned: 0


In [28]:
# Ranking Test 2: Multiple features
query = {
    "genres": ["acoustic"],
    "mood": "sad",
    "energy": "low"
}

results = rank_songs(query, top_k=5)

print("Results:", results)
print("Returned:", len(results))

Results: [('5TIXBnwtMYwgC9yc1QjFp7', np.float64(0.3645353596266682)), ('5oDDFjWMz0KiMmRzWSFfQ6', np.float64(0.3382465542666065)), ('2GwShNdcJHDLEkGO4L34lF', np.float64(0.33809544629268795)), ('6Fomnvc3pkLvUQYUkhiQPx', np.float64(0.33274246484195635)), ('0imuQm7CnK74UdorFpC2Eg', np.float64(0.3319676086456545))]
Returned: 5
